# Objetivo

Desenvolvimento do Teleutils em ambiente de teste local

# Configuração do Ambiente

## Bibliotecas

In [ ]:
# 1. Módulos da Biblioteca Padrão
import logging
from pathlib import Path

# 2. Módulos de Terceiros
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import pandas_udf
from pyspark.sql import types as T # ArrayType, StructType, StructField, StringType, BooleanType

# 3. Teletools/Teleutils
from teleutils.core.extractors import CDRParquetExtractor, CDRTextExtractor
from teleutils.core.transformers import CDRTransformer

In [ ]:
# Configuração do Pandas
# Exibir todas as colunas
pd.set_option("display.max_columns", None)

# Exibir todas as linhas
pd.set_option("display.max_rows", None)

# Evitar que o texto nas células seja cortado
pd.set_option("display.max_colwidth", None)

In [ ]:
# Configuração para exibir logs no output da célula
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)

# Reduzir ruído de bibliotecas externas (pyspark, py4j, etc.)
logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("pyspark").setLevel(logging.WARNING)

## Spark

In [ ]:
spark = (
    SparkSession
        .builder
        .appName("SessaoLocal")
        .master("local[12]")
        .config("spark.driver.memory", "24g")
        .getOrCreate()
)
spark

## Arquivos de dados

### Dados de origem

In [ ]:
folder_oi = Path("/assets/shared/cdr/cdr_bruto/Semana55/Oi/CDR7N")
_files_oi = [str(file) for file in folder_oi.glob("*.gz")]
df = pd.DataFrame(_files_oi, columns=["file_path"])
df["exchange_id"] = df["file_path"].str.extract(r"CDR7N/([A-Z0-9]{4}).")
files_oi = df.groupby("exchange_id").sample(1,random_state=42)["file_path"].to_list()
files_oi[:5]

### Arquivos intermediário e final

In [ ]:
extracted_oi = "/assets/data/cdr/teletools/db/extraidos/7n_oi_extraido.parquet"
transformed_oi = "/assets/data/cdr/teletools/db/transformados/7n_oi_transformado.parquet"

# Análise

## Dados brutos

In [ ]:
rdd = spark.sparkContext.textFile(",".join(files_oi))
df_raw = rdd.map(lambda x: (x,)).toDF(["value"])
df_raw.show(5, truncate=False)

In [ ]:
column_indices = (1,2,34,50,66,82,84,85,115,135,155,175,176,194,202,208,214,215,217,222,230,235,243,247,249,250,252,253,255,259,260,261,262,263,264,265,266,366)
column_sizes = (1,32,16,16,16,2,1,30,20,20,20,1,18,8,6,6,1,2,5,8,5,8,4,2,1,2,1,2,4,1,1,1,1,1,1,1,100,1)
column_names = ("Início do Registro","Cód_CDR","Número do terminal de origem","Número do terminal de origem modificado pelo PTS","Número do terminal de origem formatado pela AsGa","Categoria do terminal de origem ","NAI do terminal de origem (NA NAI)","Número do terminal de destino","Número do terminal de destino modificado pelo PTS","Número do terminal de destino modificado pela rede","Número do terminal de destino formatado pela AsGa","NAI do terminal de destino (NB NAI)","Número de C","Data de início da chamada, em DDMMAAAA","Hora de início da chamada, em HHMMSS","Duração da chamada, em  HHMMSS","PT","Fim de Seleção, FDS","OPC","Rota de entrada","DPC","Rota de saída","Circuito","Classe da chamada","Causa da saída","Contador de saídas parciais","Indicador de CDR de Rede Inteligente","Indicador de Serviço","Bilhetador 7IP","Indicador de falha no sequenciamento de mensagens ","Indicador de Colisão","Indicador de validação","Indicador de fluxo de Chamada","Informação de translação/portabilidade - Ass. A","Informação de translação/portabilidade - Ass. B","Informação de translação/portabilidade - Ass. C","Enchimento","Fim do Registro")

columns_expressions = [
    F.trim(
        F.regexp_replace(F.substring(F.col("value"), indice, size), "-", "")
    ).alias(name)
    for indice, size, name in zip(
        column_indices, column_sizes, column_names
    )
]
df = df_raw.select(*columns_expressions)
df.limit(5).toPandas()

In [ ]:
fill_char = "-"
if fill_char:
    print("Remove fill")
else:
    print("Nothing to remove")

In [ ]:
columns_to_keep = ['Classe da chamada','Indicador de Serviço',]
df.select(columns_to_keep).show(5)

In [ ]:
df.groupBy("Indicador de Serviço").count().show()

In [ ]:
column_indices = (1,2,34,85,194,202,208,222,235,247,255)
column_sizes = (1,32,16,30,8,6,6,8,8,2,4)
column_names = ("_tipo_registro","referencia","numero_origem","numero_destino","_data","_hora","duracao","rota_entrada","rota_saida","_resultado_chamada","bilhetador")

columns_expressions = [
    F.trim(
        F.regexp_replace(F.substring(F.col("value"), indice, size), "-", "")
    ).alias(name)
    for indice, size, name in zip(
        column_indices, column_sizes, column_names
    )
]
df = df_raw.select(*columns_expressions).filter(F.col("_tipo_registro")=="<")
df.limit(5).toPandas()

## Extração

In [ ]:
extractor = CDRTextExtractor(spark)
extractor.extract(files_oi, extracted_oi, "stfc_7n_oi")

In [ ]:
df_extracted = spark.read.parquet(extracted_oi)
df_extracted.show(5)

## Transformação

In [ ]:
transformer = CDRTransformer(spark)
transformer.transform_stfc_7n_oi(extracted_oi, transformed_oi)

In [ ]:
df_transformed = spark.read.parquet(transformed_oi)
df_transformed.limit(5).toPandas().dropna(how="all",axis=1)